In [219]:
import pandas as pd
import numpy as np
import sqlite3
from typing import Tuple

In [240]:
## Load Data ##
conn = sqlite3.connect('data/data.db')
df = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)

In [241]:
## Display Basic Stats##
print(df.info())
print(df.describe())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB
None
                open           high            low          close  \
count  397711.000000  397711.000000  397711.000000  397711.000000   
mean       46.285479      46.747660      45.816279      46.292903   
std        67.132594      67.778210      66.477178      67.141879   
min         0.000000       0.005208       0.004801       0.005208   
25%         4.209938       4.240000       4.160000       4.200000  

In [242]:
def clean_ticker_column(df: pd.DataFrame, column: str = "ticker") -> pd.DataFrame:
    """Return a copy of `df` with HTML tags stripped from the ticker column.

    Some tickers come wrapped in HTML, e.g.
        '<a target="_blank"><span style="color: red">AMGN</span></a>'
    This keeps only the text between the tags ('AMGN'), trims whitespace,
    and uppercases the result. Plain tickers like 'IBM' are unchanged.

    Args:
        df: DataFrame containing a column of ticker symbols.
        column: Name of the ticker column (default "ticker").

    Returns:
        A new DataFrame with the cleaned column; the input is not modified.
    """
    cleaned = df.copy()
    cleaned[column] = (
        cleaned[column]
        .astype("string")                           # consistent string dtype; missing values stay <NA>
        .str.replace(r"<[^>]+>", "", regex=True)    # remove every <...> HTML tag
        .str.strip()                                # drop leading/trailing whitespace
        .str.upper()                                # standardize case
    )
    return cleaned

In [243]:
## Show Column Definitions, Date Range, Ticker List ##
print('Column Names Below:')
print(df.columns.values)
print("ts is Date, Others are Self Evident")
minDate, maxDate = df['ts'].min(), df['ts'].max()
print("Date Range: " + minDate + " - " + maxDate)


Column Names Below:
['ts' 'open' 'high' 'low' 'close' 'adj_close' 'volume' 'ticker']
ts is Date, Others are Self Evident
Date Range: 1962-01-02 - 2024-12-06


In [244]:
print("Ticker List Below:")
print(len(df['ticker'].unique()))
df = clean_ticker_column(df) ## These are some messy messy tickers ##
print(df['ticker'].unique())



Ticker List Below:
150
<StringArray>
[ 'CAT',  'CVX',   'HD', 'AAPL',  'TRV',  'JNJ',  'DIS', 'CSCO',  'SHW',
   'KO',   'VZ',  'IBM',   'PG',  'HON',  'AXP',  'MMM',   'GS',   'BA',
  'CRM',  'UNH',  'NKE', 'NVDA', 'AMGN',  'MCD',    'V',  'MRK',  'WMT',
  'JPM', 'MSFT', 'AMZN']
Length: 30, dtype: string


In [245]:
def drop_duplicate_ticker_ts(df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """Remove duplicate (ticker, ts) rows, keeping the most complete row in each group.

    "Most complete" = fewest missing values across all columns. If rows tie,
    the first occurrence is kept. Returns (deduped_df, number_of_rows_removed).
    """
    key = [TICKER_COL, DATE_COL]
    out = df.assign(_n_missing=df.isna().sum(axis=1))
    # Stable sort: most complete rows first, ties keep their original order, so "first" wins
    out = out.sort_values("_n_missing", kind="stable").drop_duplicates(key, keep="first")
    return out.drop(columns="_n_missing").sort_index(), len(df) - len(out)

In [246]:
df, duplicate_count = drop_duplicate_ticker_ts(df)
df = df.sort_values(by = ['ts', 'ticker'])
print(duplicate_count)

29460


In [247]:
## Data Quality Checks: Helper Functions ##

# Column names (match your DataFrame)
DATE_COL = "ts"
TICKER_COL = "ticker"
PRICE_COLS = ["open", "high", "low", "close", "adj_close"]

REPORT_COLUMNS = ["ticker", "issue_type", "issue_date", "description", "severity"]
SEVERITY_ORDER = ["high", "medium", "low"]


def _build_issue_rows(rows: pd.DataFrame, issue_type: str, description, severity: str) -> pd.DataFrame:
    """Convert flagged rows of `df` into report-format rows.

    `description` may be a single string (same for every row) or a list/Series (one per row).
    """
    if isinstance(description, pd.Series):
        description = description.to_numpy()
    return pd.DataFrame(
        {
            "ticker": rows[TICKER_COL].to_numpy(),
            "issue_type": issue_type,
            "issue_date": pd.to_datetime(rows[DATE_COL], errors="coerce").to_numpy(),
            "description": description,
            "severity": severity,
        },
        columns=REPORT_COLUMNS,
    )


def _combine(frames: list) -> pd.DataFrame:
    """Stack issue frames, or return an empty report if there were no issues."""
    if not frames:
        return pd.DataFrame(columns=REPORT_COLUMNS)
    return pd.concat(frames, ignore_index=True)


def check_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Flag every missing (null) value in every column, one row per missing cell."""
    frames = []
    for col in df.columns:
        mask = df[col].isna()
        if mask.any():
            frames.append(
                _build_issue_rows(df[mask], "missing_value", f"Missing value in '{col}'", "high")
            )
    return _combine(frames)


def check_missing_trading_days(df: pd.DataFrame) -> pd.DataFrame:
    """Flag gaps in each ticker's time series.

    The trading calendar is the set of all dates that appear anywhere in the data.
    For each ticker, any calendar date between its first and last date that the
    ticker has no row for is flagged.
    """
    dates = pd.to_datetime(df[DATE_COL], errors="coerce")
    all_days = pd.DatetimeIndex(dates.dropna().unique()).sort_values()

    frames = []
    for ticker, ticker_dates in dates.groupby(df[TICKER_COL]):
        ticker_dates = pd.DatetimeIndex(ticker_dates.dropna().unique())
        if ticker_dates.empty:
            continue
        expected = all_days[(all_days >= ticker_dates.min()) & (all_days <= ticker_dates.max())]
        missing = expected.difference(ticker_dates)
        if len(missing):
            frames.append(
                pd.DataFrame(
                    {
                        "ticker": ticker,
                        "issue_type": "missing_trading_day",
                        "issue_date": missing,
                        "description": f"No data for {ticker} on a day other tickers traded",
                        "severity": "medium",
                    },
                    columns=REPORT_COLUMNS,
                )
            )
    return _combine(frames)


def check_ohlc_bounds(df: pd.DataFrame) -> pd.DataFrame:
    """Flag days where high < close or low > close. (Not applied to adj_close.)"""
    frames = []

    high_mask = df["high"] < df["close"]
    if high_mask.any():
        bad = df[high_mask]
        desc = [f"high ({h}) < close ({c})" for h, c in zip(bad["high"], bad["close"])]
        frames.append(_build_issue_rows(bad, "high_below_close", desc, "high"))

    low_mask = df["low"] > df["close"]
    if low_mask.any():
        bad = df[low_mask]
        desc = [f"low ({l}) > close ({c})" for l, c in zip(bad["low"], bad["close"])]
        frames.append(_build_issue_rows(bad, "low_above_close", desc, "high"))

    return _combine(frames)


def check_invalid_prices(df: pd.DataFrame) -> pd.DataFrame:
    """Flag non-finite (NaN/inf) and non-positive (<= 0) values in all price columns, including adj_close."""
    frames = []
    for col in PRICE_COLS:
        values = pd.to_numeric(df[col], errors="coerce")

        non_finite = ~np.isfinite(values)
        if non_finite.any():
            frames.append(
                _build_issue_rows(
                    df[non_finite], "non_finite_price", f"'{col}' is not finite (NaN or inf)", "high"
                )
            )

        non_positive = values <= 0  # NaN compares False, so NaNs aren't double-counted here
        if non_positive.any():
            desc = [f"'{col}' is non-positive ({v})" for v in values[non_positive]]
            frames.append(_build_issue_rows(df[non_positive], "non_positive_price", desc, "high"))

    return _combine(frames)

In [248]:
## Daily Returns and Return Outlier Check ##

def calculate_daily_returns(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of `df`, sorted by ticker and date, with a `dailyReturns` column.

    dailyReturns = close / prev_close - 1, computed within each ticker so one
    ticker's first day isn't compared to another ticker's last day.
    The first row of each ticker is NaN (no previous close).
    """
    out = df.sort_values([TICKER_COL, DATE_COL]).copy()
    prev_close = out.groupby(TICKER_COL)["close"].shift(1)
    out["dailyReturns"] = out["close"] / prev_close - 1
    return out


def check_return_outliers(df: pd.DataFrame, n_std: float = 3.0) -> pd.DataFrame:
    """Flag days where |return - mean_return| > n_std * std_return, using per-ticker mean and std.

    Requires the `dailyReturns` column from calculate_daily_returns().
    """
    if "dailyReturns" not in df.columns:
        raise ValueError("Run calculate_daily_returns(df) first: 'dailyReturns' column not found.")

    # Exclude inf so a single divide-by-zero doesn't make std infinite
    returns = df["dailyReturns"].replace([np.inf, -np.inf], np.nan)
    grouped = returns.groupby(df[TICKER_COL])
    mean = grouped.transform("mean")
    std = grouped.transform("std")

    mask = (returns - mean).abs() > n_std * std
    if not mask.any():
        return _combine([])

    desc = [
        f"Daily return {r:.2%} is more than {n_std:g} std from mean ({m:.2%}, std {s:.2%})"
        for r, m, s in zip(returns[mask], mean[mask], std[mask])
    ]
    return _build_issue_rows(df[mask], "return_outlier", desc, "medium")

In [249]:
## Data Quality Report Wrapper ##

def create_data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """Run all data quality checks and return a single combined issue report.

    Columns: ticker | issue_type | issue_date | description | severity
    Sorted by severity (high -> low), then ticker, then date.
    """
    with_returns = calculate_daily_returns(df)

    report = pd.concat(
        [
            check_missing_values(df),  # original df, so first-day NaN returns aren't flagged
            check_missing_trading_days(df),
            check_ohlc_bounds(df),
            check_invalid_prices(df),
            check_return_outliers(with_returns),
        ],
        ignore_index=True,
    )

    report["severity"] = pd.Categorical(report["severity"], categories=SEVERITY_ORDER, ordered=True)
    return report.sort_values(["severity", "ticker", "issue_date"], ignore_index=True)

In [250]:
def test_date_verifier(df: pd.DataFrame, original_report: pd.DataFrame, ticker: str, date) -> pd.DataFrame:
    """Remove one ticker/date row, rerun the data quality report, and show what changed.

    Demonstrates that check_missing_trading_days catches a gap we create on purpose:
    the new report should contain a 'missing_trading_day' row for (ticker, date)
    that wasn't in the original report.

    Args:
        df: The (cleaned) price data used to build original_report.
        original_report: Output of create_data_quality_report(df).
        ticker: Ticker whose row to remove, e.g. "AAPL".
        date: Date to remove, e.g. "2020-03-16" (string or datetime).

    Returns:
        The rows that are in the new report but not in the original (the anti-join).
    """
    target_date = pd.to_datetime(date)
    row_dates = pd.to_datetime(df[DATE_COL], errors="coerce")
    is_ticker = df[TICKER_COL] == ticker
    to_remove = is_ticker & (row_dates == target_date)

    if not to_remove.any():
        print(f"No row found for {ticker} on {target_date.date()}; nothing removed.")
        return pd.DataFrame(columns=REPORT_COLUMNS)

    # The gap check only looks between a ticker's first and last dates
    ticker_dates = row_dates[is_ticker]
    if target_date in (ticker_dates.min(), ticker_dates.max()):
        print(f"Warning: {target_date.date()} is {ticker}'s first/last date, "
              "so removing it shortens the series rather than creating a gap. Pick an interior date.")

    # Show the ticker's issues BEFORE removal, for comparison
    before = original_report[(original_report[TICKER_COL] == ticker) & (original_report['issue_date'] == date)]
    print(f"{ticker} issues BEFORE removal ({len(before)}):")
    print(before.to_string(index=False) if len(before) else "  (none)")
    print()
    # Remove the row from a copy and rerun the full report
    modified = df.loc[~to_remove].copy()
    new_report = create_data_quality_report(modified)

    # Anti-join in both directions: outer merge on every column, then use the merge indicator.
    # Severity is cast to str so the categorical columns compare cleanly.
    merged = (
        new_report.astype({"severity": str})
        .merge(original_report.astype({"severity": str}), on=REPORT_COLUMNS, how="outer", indicator=True)
    )
    added = merged[merged["_merge"] == "left_only"].drop(columns="_merge")
    removed = merged[merged["_merge"] == "right_only"].drop(columns="_merge")

    print(f"Removed {int(to_remove.sum())} row(s) for {ticker} on {target_date.date()}\n")
    print(f"NEW issues after removal ({len(added)}):")
    print(added.to_string(index=False) if len(added) else "  (none)")
    print(f"\nIssues that DISAPPEARED after removal ({len(removed)}):")

In [251]:
## Resolution Logic v2: OHLC Bounds, adj_close Repair, Repair Validation, Quarantine, Extreme Returns, Wrapper ##
#
# NOTE ON MY CHOICES
# Impossible price relationships: I chose Option 1, replace with NaN and forward-fill.
#   On any impossible bar (high below open/close/low, or low above open/close), the whole OHLC
#   bar and adj_close are set to NaN and forward-filled from the previous day, so the replacement
#   comes from one internally consistent bar. Rows with no prior bar are quarantined.
# adj_close: NOT forward-filled as a price. Forward-filling adj_close while close moves breaks the
#   adjustment factor (adj_close / close). Instead, missing, impossible (adj_close > close) and spiked
#   adj_close values are rebuilt as close x the factor from the nearest valid days, and only when the
#   valid factors before and after the gap agree (so no dividend/split happened inside the gap).
#   Source of the factor: this ticker's own adj_close / close on those neighboring days in ohlc_hw.
#   Gaps where the factors disagree (possible corporate action) are quarantined, not guessed.
# Extreme returns (> 3 std from the ticker's mean): CONFIRMED data errors are interpolated, and only
#   between two valid neighbors. An error is confirmed by either (a) a one-day spike that reverses
#   while the market is calm, or (b) my rule that an otherwise-unexplained move > 10 std must be an error.
#   Otherwise-unexplained moves are bucketed by size: 3-5 std "big event, no error detected",
#   5-10 std "no error, > 5 std". VIX high days (market-wide moves) are always labeled vix_high_day,
#   whatever their size. Splits, VIX high days, possible mergers, and unexplained events are kept and
#   labeled, with the evidence or the stated uncertainty recorded in `outlier_evidence`.
#
# This cell replaces resolve_missing_values, resolve_ohlc_bounds, quarantine_unresolved and
# resolve_data_issues from the cell above.

from typing import Dict

ERROR_TYPES = ["data_error", "error_gt_10_std"]   # outlier types treated as confirmed errors (repaired)

RAW_PRICE_COLS = ["open", "high", "low", "close"]   # OHLC only (adj_close handled separately)

def resolve_invalid_prices(df: pd.DataFrame) -> pd.DataFrame:
    """Treat non-positive (<= 0) or non-finite (inf) prices as missing.

    Sets them to NaN so resolve_missing_values() fills them like any other gap.
    Applies to all price columns, including adj_close.
    Adds flag_price_invalid = True on any row where a price was nulled.
    """
    out = df.copy()
    out["flag_price_invalid"] = False
    for col in PRICE_COLS:
        values = pd.to_numeric(out[col], errors="coerce")
        # notna() so values that were already missing aren't counted as "invalid"
        invalid = values.notna() & (~np.isfinite(values) | (values <= 0))
        out.loc[invalid, col] = np.nan
        out["flag_price_invalid"] |= invalid
    return out


def drop_all_price_missing(df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """Drop rows where ALL raw price columns (open, high, low, close) are missing.

    Run this before any filling so empty rows aren't filled with made-up values.
    Returns (remaining_df, number_of_rows_dropped).
    """
    all_missing = df[RAW_PRICE_COLS].isna().all(axis=1)
    return df.loc[~all_missing].copy(), int(all_missing.sum())


def raw_bounds_violations(df: pd.DataFrame) -> pd.Series:
    """True where an OHLC bar is impossible: high below open/close/low, or low above open/close."""
    o, h, l, c = (df[col] for col in RAW_PRICE_COLS)
    return (h < o) | (h < c) | (l > o) | (l > c) | (h < l)


def resolve_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing OHLC and volume within each ticker. adj_close is left for resolve_adj_close().

    - open/high/low/close: forward-fill (carry the last known price)
    - volume: fill with 0 and set flag_volume_filled. Unknown volume doesn't
      mean no trading, so the flag records that the 0 isn't real.

    Values with no earlier value to carry (e.g. a ticker's first row) stay NaN
    and are caught by quarantine_unresolved().
    """
    out = df.sort_values([TICKER_COL, DATE_COL]).copy()

    price_was_missing = out[RAW_PRICE_COLS].isna()
    out[RAW_PRICE_COLS] = out.groupby(TICKER_COL)[RAW_PRICE_COLS].ffill()
    out["flag_price_ffilled"] = (price_was_missing & out[RAW_PRICE_COLS].notna()).any(axis=1)

    out["flag_volume_filled"] = out["volume"].isna()
    out["volume"] = out["volume"].fillna(0)

    return out


def resolve_adj_close(df: pd.DataFrame, spike_tol: float = 0.02, agree_tol: float = 0.005,
                      window: int = 5) -> pd.DataFrame:
    """Rebuild bad adj_close values from the validated close and a trusted, date-specific factor.

    factor = adj_close / close. For backward-adjusted prices it is <= 1 and steps up only at
    dividends/splits. An adj_close is BAD if it is:
      - missing,
      - impossible: factor > 1 (adj_close above close), or
      - a spike: factor differs from the ticker's centered rolling median factor by > spike_tol.
    A bad row is rebuilt as close x the last valid factor, but ONLY if the nearest valid factors
    before and after it agree within agree_tol, which means no corporate action inside the gap.
    Otherwise adj_close is left NaN and the row is quarantined (never forward-filled across an
    unverified corporate action). Adds flag_adj_close_rebuilt.
    """
    out = df.sort_values([TICKER_COL, DATE_COL]).copy()
    t = out[TICKER_COL]
    factor = out["adj_close"] / out["close"]
    median_factor = factor.groupby(t).transform(
        lambda s: s.rolling(window, center=True, min_periods=3).median()
    )
    bad = factor.isna() | (factor > 1 + 1e-3) | ((factor / median_factor - 1).abs() > spike_tol)

    good_factor = factor.where(~bad)
    prev_factor = good_factor.groupby(t).ffill()     # nearest valid factor before
    next_factor = good_factor.groupby(t).bfill()     # nearest valid factor after
    fixable = bad & ((prev_factor / next_factor - 1).abs() <= agree_tol)

    out.loc[fixable, "adj_close"] = out.loc[fixable, "close"] * prev_factor[fixable]
    out.loc[bad & ~fixable, "adj_close"] = np.nan    # not defensible to rebuild -> quarantine
    out["flag_adj_close_rebuilt"] = fixable
    return out



def resolve_ohlc_bounds(df: pd.DataFrame) -> pd.DataFrame:
    """Fix impossible OHLC bars by setting them to NaN and forward-filling from the previous day.

    CHOICE (mine): Option 1 from the assignment, replace with NaN and then forward-fill.
    The whole bar (open, high, low, close) and adj_close are nulled together, so the
    replacement values all come from one consistent prior bar. Values that were ALREADY
    missing (e.g. adj_close that resolve_adj_close couldn't rebuild) stay NaN for quarantine.
    Adds flag_hl_ffilled.
    """
    out = df.sort_values([TICKER_COL, DATE_COL]).copy()
    bad = raw_bounds_violations(out)
    was_missing = out[PRICE_COLS].isna()            # left for quarantine; must not be filled here
    out["flag_hl_ffilled"] = bad
    out.loc[bad, PRICE_COLS] = np.nan
    out[PRICE_COLS] = out.groupby(TICKER_COL)[PRICE_COLS].ffill().mask(was_missing)
    return out


def resolve_missing_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Insert a row for every missing trading day in each ticker's series.

    Uses the same calendar as check_missing_trading_days(): every date that
    appears anywhere in the data, between each ticker's first and last date.
    A filled-in day is a flat bar copied from the last REAL day before it:
        open = high = low = close = that day's close
        adj_close = that day's close x that day's factor (NaN if its adj_close is NaN -> quarantine)
        volume    = that day's volume
    Adds flag_date_filled = True on inserted rows.
    """
    out = df.copy()
    out[DATE_COL] = pd.to_datetime(out[DATE_COL], errors="coerce")
    out["flag_date_filled"] = False
    all_days = pd.DatetimeIndex(out[DATE_COL].dropna().unique()).sort_values()

    pieces = []
    for ticker, group in out.groupby(TICKER_COL):
        group = group.set_index(DATE_COL).sort_index()
        expected = all_days[(all_days >= group.index.min()) & (all_days <= group.index.max())]
        missing = expected.difference(group.index)

        new_rows = pd.DataFrame({TICKER_COL: ticker, "flag_date_filled": True}, index=missing)
        combined = pd.concat([group, new_rows]).sort_index(kind="stable")
        is_new = combined["flag_date_filled"].astype(bool).to_numpy()

        # Position of the last real row at or before each row (the first row is always real)
        last_real = pd.Series(np.where(is_new, np.nan, np.arange(len(combined)))).ffill().astype(int).to_numpy()
        last_close = combined["close"].to_numpy()[last_real]
        last_factor = (combined["adj_close"] / combined["close"]).to_numpy()[last_real]

        for col in RAW_PRICE_COLS:
            combined.loc[is_new, col] = last_close[is_new]
        combined.loc[is_new, "adj_close"] = last_close[is_new] * last_factor[is_new]
        combined.loc[is_new, "volume"] = combined["volume"].to_numpy()[last_real][is_new]

        pieces.append(combined)

    result = pd.concat(pieces).rename_axis(DATE_COL).reset_index()

    for col in result.columns:
        if col.startswith("flag_"):
            result[col] = result[col].fillna(False).astype(bool)
    if result["volume"].notna().all():
        result["volume"] = result["volume"].astype("int64")

    return result[[c for c in out.columns]]   # keep the original column order


def validate_repairs(df: pd.DataFrame, step: str, factor_jump_tol: float = 0.5) -> Dict[str, int]:
    """Recheck raw OHLC bounds and adjusted-price consistency, and print a one-line summary.

    Raw:      impossible bars, missing prices, non-finite or non-positive prices.
    Adjusted: factor = adj_close / close must be finite and > 0.
              adj_close > close is flagged, since backward-adjusted prices shouldn't exceed the raw price.
              Day-over-day factor jumps > factor_jump_tol are flagged; they're only legitimate at splits.
    """
    df = df.sort_values([TICKER_COL, DATE_COL])
    prices = df[PRICE_COLS].apply(pd.to_numeric, errors="coerce")
    factor = prices["adj_close"] / prices["close"]
    prev_factor = factor.groupby(df[TICKER_COL]).shift(1)

    counts = {
        "bounds_violations": int(raw_bounds_violations(prices).sum()),
        "missing_prices": int(prices.isna().any(axis=1).sum()),
        "non_finite_or_non_positive": int(((prices.notna() & ~np.isfinite(prices)) | (prices <= 0)).any(axis=1).sum()),
        "bad_adj_factor": int((factor.notna() & (~np.isfinite(factor) | (factor <= 0))).sum()),
        "adj_above_close": int((factor > 1 + 1e-4).sum()),
        "adj_factor_jumps": int(((factor / prev_factor - 1).abs() > factor_jump_tol).sum()),
    }
    print(f"[check after {step}] " + ", ".join(f"{k}={v}" for k, v in counts.items()))
    return counts


def quarantine_unresolved(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Split off rows that still have missing prices/volume after every repair.

    Returns (clean_df, quarantined_df). Quarantined rows get a `quarantine_reason`.
    """
    missing = df[PRICE_COLS + ["volume"]].isna()
    unresolved = missing.any(axis=1)
    only_adj = missing["adj_close"] & ~missing.drop(columns="adj_close").any(axis=1)

    quarantined = df.loc[unresolved].copy()
    quarantined["quarantine_reason"] = np.where(
        only_adj[unresolved],
        "adj_close not rebuildable: valid factors before/after disagree or are missing (possible corporate action)",
        "missing price/volume with no earlier value to forward-fill",
    )
    return df.loc[~unresolved].copy(), quarantined


def classify_return_outliers(df: pd.DataFrame, n_std: float = 3.0, round_tol: float = 0.03,
                             reversal_tol: float = 0.5, market_z: float = 2.0, market_share: float = 0.3,
                             window: int = 20, pinned_ratio: float = 0.33,
                             no_error_flag_std: float = 5.0, must_be_error_std: float = 10.0) -> pd.DataFrame:
    """Classify each extreme return (|return - mean| > n_std * std, per ticker) and record evidence.

    Adds `outlier_type` and `outlier_evidence` (NaN on normal days). Labels, in priority order:
      data_error      - CONFIRMED error: a one-day spike that reverses the next day (close returns
                        within reversal_tol * |return| of the prior close) while the market was calm
      error_reversal  - the snap-back day after a data_error
      split           - raw close changed by a round multiple (x1.5, x2, x2.5, x3, ...) AND
                        adj_close was continuous that day
      possible_split  - round multiple, but adj_close jumped too or is missing: NOT confirmed
      vix_high_day    - >= market_share of the OTHER tickers also moved > market_z std that date
                        (a market-wide stress proxy; no actual VIX data is used). Takes priority
                        over the size buckets below.
      possible_merger - UNCONFIRMED pattern: volatility collapsed afterward (price pinned, as with a
                        pending cash deal) or the ticker's data ends within `window` days
      Otherwise-unexplained moves, bucketed by size:
      error_gt_10_std             - |z| > must_be_error_std: treated as a confirmed error (my rule)
      no_error_gt_5_std           - no_error_flag_std < |z| <= must_be_error_std: kept, no error detected
      big_event_no_error_detected - n_std < |z| <= no_error_flag_std: kept, cause unknown
    Returns a copy sorted by ticker and date.
    """
    out = calculate_daily_returns(df)
    t = out[TICKER_COL]
    by_ticker = out.groupby(TICKER_COL)

    # Extreme days (same rule as check_return_outliers)
    ret = out["dailyReturns"].replace([np.inf, -np.inf], np.nan)
    z = (ret - ret.groupby(t).transform("mean")) / ret.groupby(t).transform("std")
    is_outlier = z.abs() > n_std

    # Data error evidence: the spike reverses the next day
    prev_close = by_ticker["close"].shift(1)
    next_close = by_ticker["close"].shift(-1)
    next_ret = ret.groupby(t).shift(-1)
    round_trip = (next_close / prev_close - 1).abs()
    reverses = (np.sign(next_ret) == -np.sign(ret)) & (round_trip < reversal_tol * ret.abs())

    # Market-wide stress: how many OTHER tickers also had a big move that date
    big = (z.abs() > market_z).astype(int)
    traded = z.notna().astype(int)
    dates = out[DATE_COL]
    others_big = big.groupby(dates).transform("sum") - big
    others = traded.groupby(dates).transform("sum") - traded
    market_day = (others >= 5) & (others_big / others.where(others > 0) >= market_share)

    # Split evidence: round-multiple price change, adj_close continuity, volume scaling
    ratio = prev_close / out["close"]
    ratio = ratio.where(ratio >= 1, 1 / ratio)              # normal and reverse splits both give ratio >= 1
    nearest = (ratio * 2).round() / 2                       # nearest half multiple: 1.5, 2, 2.5, 3, ...
    round_multiple = (nearest >= 1.5) & ((ratio - nearest).abs() <= round_tol * nearest)
    adj_ret = out["adj_close"] / by_ticker["adj_close"].shift(1) - 1
    adj_z = (adj_ret - adj_ret.groupby(t).transform("mean")) / adj_ret.groupby(t).transform("std")
    adj_continuous = adj_z.abs() <= n_std
    vol = out["volume"].astype(float)
    vol_before = vol.groupby(t).transform(lambda s: s.rolling(window, min_periods=5).median().shift(1))
    vol_after = vol.groupby(t).transform(lambda s: s[::-1].rolling(window, min_periods=5).median()[::-1])
    vol_change = vol_after / vol_before.where(vol_before > 0)
    expected_vol = ratio.where(out["close"] < prev_close, 1 / ratio)   # forward split -> volume should rise

    # Merger pattern: price pinned afterward, or the series ends soon after
    pre_vol = ret.groupby(t).transform(lambda s: s.rolling(window, min_periods=10).std().shift(1))
    post_vol = ret.groupby(t).transform(lambda s: s[::-1].rolling(window, min_periods=10).std()[::-1].shift(-1))
    pinned = post_vol < pinned_ratio * pre_vol
    ends_early = by_ticker[DATE_COL].transform("max") < out[DATE_COL].max()
    ends_soon = ends_early & (by_ticker.cumcount(ascending=False) < window)

    is_error = is_outlier & reverses & ~market_day
    is_reversal = is_outlier & ~is_error & is_error.groupby(t).shift(1, fill_value=False)

    # np.select picks the FIRST matching condition, so vix_high_day wins over the size buckets
    out["outlier_type"] = np.select(
        [is_error, is_reversal,
         is_outlier & round_multiple & adj_continuous,
         is_outlier & round_multiple,
         is_outlier & market_day,
         is_outlier & (pinned | ends_soon),
         is_outlier & (z.abs() > must_be_error_std),
         is_outlier & (z.abs() > no_error_flag_std),
         is_outlier],
        ["data_error", "error_reversal", "split", "possible_split",
         "vix_high_day", "possible_merger",
         "error_gt_10_std", "no_error_gt_5_std", "big_event_no_error_detected"],
        default="",
    )
    out["outlier_type"] = out["outlier_type"].replace("", np.nan)

    # Record the evidence (or the uncertainty) behind each label
    no_pattern = "no reversal, round-multiple, market-wide, or merger pattern"
    evidence = {}
    for i in out.index[is_outlier]:
        kind = out.at[i, "outlier_type"]
        base = f"return {ret.at[i]:+.1%} (z={z.at[i]:+.1f})"
        mkt = f"{int(others_big.at[i])}/{int(others.at[i])} other tickers moved > {market_z:g} std"
        if kind == "data_error":
            note = f"spike reversed next day (close back within {round_trip.at[i]:.1%} of prior close); {mkt}"
        elif kind == "error_reversal":
            note = "snap-back from the previous day's data error"
        elif kind in ("split", "possible_split"):
            vol_note = (f"volume x{vol_change.at[i]:.2f} vs ~x{expected_vol.at[i]:.2f} expected"
                        if pd.notna(vol_change.at[i]) else "volume change unavailable")
            if kind == "split":
                note = (f"raw close moved ~{nearest.at[i]:g}x (actual {ratio.at[i]:.3f}x) while adj_close "
                        f"was continuous ({adj_ret.at[i]:+.1%}); {vol_note}")
            else:
                adj_note = "adj_close missing" if pd.isna(adj_ret.at[i]) else f"adj_close also moved {adj_ret.at[i]:+.1%}"
                note = f"raw close moved ~{nearest.at[i]:g}x but {adj_note}, so split NOT confirmed; {vol_note}"
        elif kind == "vix_high_day":
            note = f"VIX high day, market-wide move: {mkt} (proxy; no VIX data used)"
        elif kind == "possible_merger":
            reasons = []
            if pinned.at[i]:
                reasons.append(f"next-{window}-day volatility fell to {post_vol.at[i] / pre_vol.at[i]:.0%} of prior (price pinned)")
            if ends_soon.at[i]:
                reasons.append(f"ticker's data ends within {window} days")
            note = "possible merger/acquisition, UNCONFIRMED (no corporate-action data): " + "; ".join(reasons)
        elif kind == "error_gt_10_std":
            note = (f"move > {must_be_error_std:g} std with {no_pattern}, and {mkt}; "
                    f"treated as an error under the > {must_be_error_std:g} std rule")
        elif kind == "no_error_gt_5_std":
            note = f"no error detected, move > {no_error_flag_std:g} std: {no_pattern}; cause unknown"
        else:
            note = f"big stock event day, no error detected: {no_pattern}; cause unknown"
        evidence[i] = f"{base}; {note}"
    out["outlier_evidence"] = pd.Series(evidence, dtype=object)

    return out.drop(columns="dailyReturns")


def resolve_return_outliers(df: pd.DataFrame, **classify_kwargs) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Repair only CONFIRMED errors (ERROR_TYPES); keep and flag every other extreme return.

    An error row (data_error or error_gt_10_std) is interpolated (OHLC + adj_close, linear,
    within its sorted ticker) only when BOTH neighbors exist and aren't errors themselves.
    Interpolating close and adj_close with the same weights keeps the factor between its
    neighbors' factors. Nothing is extrapolated at the start or end of a series. Errors without
    two valid neighbors are quarantined with a reason.
    Returns (clean_df, quarantined_df).
    """
    out = classify_return_outliers(df, **classify_kwargs)
    t = out[TICKER_COL]
    is_error = out["outlier_type"].isin(ERROR_TYPES)

    prev_ok = is_error.groupby(t).shift(1).eq(False)    # NaN (no neighbor) counts as not OK
    next_ok = is_error.groupby(t).shift(-1).eq(False)
    can_fix = is_error & prev_ok & next_ok
    cannot_fix = is_error & ~can_fix

    out.loc[can_fix, PRICE_COLS] = np.nan
    out[PRICE_COLS] = out.groupby(TICKER_COL)[PRICE_COLS].transform(
        lambda s: s.interpolate(limit_area="inside")    # between valid neighbors only, never past the ends
    )
    out.loc[can_fix, "outlier_evidence"] += "; prices interpolated from neighboring days"
    out["flag_outlier_interpolated"] = can_fix
    out["flag_outlier_kept"] = out["outlier_type"].notna() & ~is_error

    quarantined = out[cannot_fix].copy()
    quarantined["quarantine_reason"] = "confirmed error without valid neighbors on both sides; interpolation not defensible"
    return out[~cannot_fix].copy(), quarantined


def resolve_data_issues(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run every resolve step in order, rechecking bounds and adjusted consistency after each.

      1. invalid prices -> NaN
      2. drop rows where all OHLC are missing (before any filling)
      3. fill missing OHLC (ffill) and volume (0 + flag)
      3b. rebuild missing/impossible/spiked adj_close from close x a verified neighboring factor
      4. fill missing trading days (flat bar at the previous close, previous adj_close)
      5. impossible OHLC bars -> NaN + ffill (my choice)
      6. quarantine anything still unresolved
      7. extreme returns: interpolate confirmed errors only; flag the rest with evidence

    Returns (clean_df, quarantined_df). Every quarantined row has a quarantine_reason.
    """
    out = resolve_invalid_prices(df)
    validate_repairs(out, "1 invalid prices -> NaN")
    out, n_dropped = drop_all_price_missing(out)
    validate_repairs(out, "2 drop all-missing rows")
    out = resolve_missing_values(out)
    validate_repairs(out, "3 fill missing OHLC/volume")
    out = resolve_adj_close(out)
    validate_repairs(out, "3b rebuild adj_close")
    out = resolve_missing_dates(out)
    validate_repairs(out, "4 fill missing dates")
    out = resolve_ohlc_bounds(out)
    validate_repairs(out, "5 OHLC bounds (NaN + ffill)")
    clean, q_missing = quarantine_unresolved(out)
    validate_repairs(clean, "6 quarantine unresolved")
    clean, q_outliers = resolve_return_outliers(clean)
    validate_repairs(clean, "7 extreme returns")

    quarantined = pd.concat([q_missing, q_outliers], ignore_index=True)

    print("\nRepair summary")
    print(f"  Rows dropped (all OHLC missing): {n_dropped}")
    for col in [c for c in clean.columns if c.startswith("flag_")]:
        print(f"  {col}: {int(clean[col].sum())}")
    print("  Outlier classification:")
    print(clean["outlier_type"].value_counts().to_string())
    print(f"  Rows quarantined: {len(quarantined)}")
    if len(quarantined):
        print(quarantined["quarantine_reason"].value_counts().to_string())

    return clean.reset_index(drop=True), quarantined

In [252]:
data_quality_report = create_data_quality_report(df)
df_cleaned, df_quarentined = resolve_data_issues(df)
print('Low')
print(data_quality_report[data_quality_report['severity'] =='low'])
print('Medium')
print(data_quality_report[data_quality_report['severity'] =='high'])
print('High')
print(data_quality_report[data_quality_report['severity'] =='medium'])



[check after 1 invalid prices -> NaN] bounds_violations=0, missing_prices=18379, non_finite_or_non_positive=0, bad_adj_factor=0, adj_above_close=10348, adj_factor_jumps=19384
[check after 2 drop all-missing rows] bounds_violations=0, missing_prices=18379, non_finite_or_non_positive=0, bad_adj_factor=0, adj_above_close=10348, adj_factor_jumps=19384
[check after 3 fill missing OHLC/volume] bounds_violations=0, missing_prices=18379, non_finite_or_non_positive=0, bad_adj_factor=0, adj_above_close=10348, adj_factor_jumps=19384
[check after 3b rebuild adj_close] bounds_violations=0, missing_prices=928, non_finite_or_non_positive=0, bad_adj_factor=0, adj_above_close=0, adj_factor_jumps=0
[check after 4 fill missing dates] bounds_violations=0, missing_prices=930, non_finite_or_non_positive=0, bad_adj_factor=0, adj_above_close=0, adj_factor_jumps=0
[check after 5 OHLC bounds (NaN + ffill)] bounds_violations=0, missing_prices=930, non_finite_or_non_positive=0, bad_adj_factor=0, adj_above_close=0

In [253]:
df_cleaned_values = df_cleaned[df.columns]
cleaned_quality_report = create_data_quality_report(df_cleaned_values)




In [254]:
print('Low')
print(cleaned_quality_report[cleaned_quality_report['severity'] =='low'])
print('Medium')
print(cleaned_quality_report[cleaned_quality_report['severity'] =='high'])
print('High')
print(cleaned_quality_report[cleaned_quality_report['severity'] =='medium'])

Low
Empty DataFrame
Columns: [ticker, issue_type, issue_date, description, severity]
Index: []
Medium
Empty DataFrame
Columns: [ticker, issue_type, issue_date, description, severity]
Index: []
High
     ticker      issue_type issue_date  \
0      AAPL  return_outlier 1980-12-26   
1      AAPL  return_outlier 1981-04-02   
2      AAPL  return_outlier 1981-09-25   
3      AAPL  return_outlier 1982-03-19   
4      AAPL  return_outlier 1982-11-22   
...     ...             ...        ...   
5903    WMT  return_outlier 2022-07-26   
5904    WMT  return_outlier 2022-11-15   
5905    WMT  return_outlier 2023-11-16   
5906    WMT  return_outlier 2024-05-16   
5907    WMT  return_outlier 2024-08-15   

                                            description severity  
0     Daily return 9.23% is more than 3 std from mea...   medium  
1     Daily return 8.77% is more than 3 std from mea...   medium  
2     Daily return -12.98% is more than 3 std from m...   medium  
3     Daily return 9.02% is m

In [255]:
## Post-Cleaning Verification ##

def verify_prices(df: pd.DataFrame) -> Dict[str, int]:
    """Count missing, non-finite, and non-positive values across all price columns (per cell)."""
    p = df[PRICE_COLS]
    return {"missing_price": int(p.isna().sum().sum()),
            "non_finite_price": int((p.notna() & ~np.isfinite(p)).sum().sum()),
            "non_positive_price": int((p <= 0).sum().sum())}


def verify_dates(df: pd.DataFrame) -> Dict[str, int]:
    """Count unparseable dates and duplicate (ticker, ts) rows."""
    ts = pd.to_datetime(df[DATE_COL], errors="coerce")
    return {"invalid_date": int(ts.isna().sum()),
            "duplicate_ticker_ts": int(pd.DataFrame({"t": df[TICKER_COL], "ts": ts}).duplicated().sum())}


def verify_volume(df: pd.DataFrame) -> Dict[str, int]:
    """Count missing/non-finite and negative volume."""
    v = pd.to_numeric(df["volume"], errors="coerce")
    return {"non_finite_volume": int((~np.isfinite(v)).sum()),   # NaN counts as non-finite
            "negative_volume": int((v < 0).sum())}


def verify_price_consistency(df: pd.DataFrame) -> Dict[str, int]:
    """Check OHLC bounds and adj_close consistency.

    Backward-adjusted data has factor = adj_close / close <= 1, and the factor only steps
    UP over time (at dividends/splits), so a day-over-day decrease is inconsistent.
    """
    d = df.sort_values([TICKER_COL, DATE_COL])
    factor = d["adj_close"] / d["close"]
    prev_factor = factor.groupby(d[TICKER_COL]).shift(1)
    return {"ohlc_bounds_violation": int(raw_bounds_violations(d).sum()),
            "adj_close_above_close": int((factor > 1 + 1e-3).sum()),
            "adj_factor_decrease": int((factor < prev_factor * (1 - 1e-3)).sum())}


def verify_cleaned_data(df_cleaned: pd.DataFrame) -> pd.Series:
    """Run every verification and print a concise report. Returns the counts."""
    report = pd.Series({**verify_prices(df_cleaned), **verify_dates(df_cleaned),
                        **verify_volume(df_cleaned), **verify_price_consistency(df_cleaned)}, name="issues")
    print(report.to_string())
    print("PASS: no critical issues" if report[report.index != 'adj_factor_decrease'].sum() == 0 else f"FAIL: {int(report.sum())} total issues")
    return report



In [256]:
df_verification = verify_cleaned_data(df)
df_verification = verify_cleaned_data(df_cleaned_values)

missing_price            18374
non_finite_price             0
non_positive_price           5
invalid_date                 0
duplicate_ticker_ts          0
non_finite_volume            0
negative_volume              0
ohlc_bounds_violation        5
adj_close_above_close    10348
adj_factor_decrease      35317
FAIL: 64049 total issues
missing_price                0
non_finite_price             0
non_positive_price           0
invalid_date                 0
duplicate_ticker_ts          0
non_finite_volume            0
negative_volume              0
ohlc_bounds_violation        0
adj_close_above_close        0
adj_factor_decrease      27615
PASS: no critical issues


In [257]:
df_verification

missing_price                0
non_finite_price             0
non_positive_price           0
invalid_date                 0
duplicate_ticker_ts          0
non_finite_volume            0
negative_volume              0
ohlc_bounds_violation        0
adj_close_above_close        0
adj_factor_decrease      27615
Name: issues, dtype: int64

In [258]:
test_date_verifier(df, data_quality_report, 'AAPL', '2020-03-31')

AAPL issues BEFORE removal (2):
ticker       issue_type issue_date                            description severity
  AAPL    missing_value 2020-03-31           Missing value in 'adj_close'     high
  AAPL non_finite_price 2020-03-31 'adj_close' is not finite (NaN or inf)     high

Removed 1 row(s) for AAPL on 2020-03-31

NEW issues after removal (1):
ticker          issue_type issue_date                                    description severity
  AAPL missing_trading_day 2020-03-31 No data for AAPL on a day other tickers traded   medium

Issues that DISAPPEARED after removal (2):


In [259]:
ohlc_hw_cleaned = df_cleaned_values

df_cleaned.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
conn.commit()